# Heart Disease Prediction - Model Training
## MLOps Assignment 1

This notebook demonstrates:
- Feature engineering and preprocessing
- Training multiple classification models
- Hyperparameter tuning with GridSearchCV
- Model evaluation with cross-validation
- MLflow experiment tracking

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from src.data_processing.preprocessor import load_data, preprocess_data
from src.models.train import train_models, evaluate_model

warnings.filterwarnings('ignore')
%matplotlib inline

print('Libraries imported successfully!')

## 1. Load and Preprocess Data

In [ ]:
# Load dataset
df = load_data('../data/heart_disease.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Preprocess and split data
X_train, X_test, y_train, y_test, preprocessor = preprocess_data(df, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTarget distribution in training set:")
print(pd.Series(y_train).value_counts())

# Save preprocessor
preprocessor.save('../models/preprocessor.pkl')
print("\n✓ Preprocessor saved")

## 2. Train Multiple Models with MLflow Tracking

In [ ]:
# Train models with experiment tracking
results = train_models(
    X_train, X_test, y_train, y_test,
    experiment_name="heart-disease-prediction",
    models_dir="../models"
)

print("\n" + "="*60)
print("Model Training Completed!")
print("="*60)

## 3. Compare Model Performance

In [ ]:
# Extract metrics for comparison
model_names = [name for name in results.keys() if name not in ['best_model_name', 'best_model_score']]

comparison_data = []
for name in model_names:
    metrics = results[name]['metrics']
    comparison_data.append({
        'Model': name,
        'Test Accuracy': metrics['test_accuracy'],
        'Test Precision': metrics['test_precision'],
        'Test Recall': metrics['test_recall'],
        'Test F1': metrics['test_f1'],
        'Test ROC-AUC': metrics['test_roc_auc'],
        'CV ROC-AUC': metrics['cv_roc_auc_mean']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Test ROC-AUC', ascending=False)

print("\nModel Performance Comparison:")
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics_to_plot = ['Test Accuracy', 'Test Precision', 'Test Recall', 'Test ROC-AUC']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    comparison_df_sorted = comparison_df.sort_values(metric, ascending=True)
    
    ax.barh(comparison_df_sorted['Model'], comparison_df_sorted[metric], 
           color=sns.color_palette('viridis', len(model_names)))
    ax.set_xlabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'Model Comparison - {metric}', fontsize=14, fontweight='bold')
    ax.set_xlim([0, 1])
    
    # Add value labels
    for i, v in enumerate(comparison_df_sorted[metric]):
        ax.text(v + 0.01, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.savefig('../screenshots/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Best Model Summary

In [ ]:
# Display best model information
best_model_name = results['best_model_name']
best_model_score = results['best_model_score']

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"ROC-AUC Score: {best_model_score:.4f}")
print(f"{'='*60}")

print(f"\nBest Hyperparameters:")
for param, value in results[best_model_name]['best_params'].items():
    print(f"  {param}: {value}")

print(f"\nFull Metrics:")
for metric, value in results[best_model_name]['metrics'].items():
    print(f"  {metric}: {value:.4f}")